# Preprocess Data

BTS becomes a flight-level table with UTC schedule and actual timestamps. SFO NOAA observations become numeric weather fields. FAA records are standardized to one row per tail.

In [1]:
from __future__ import annotations

from pathlib import Path
from types import SimpleNamespace
import json
import pandas as pd
import airportsdata
import numpy as np
import pyarrow.parquet as pq

ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

CONFIG_PATH = ROOT / "config" / "project_config.json"

with CONFIG_PATH.open(encoding="utf-8") as file:
    CONFIG = json.load(file)

PATHS = SimpleNamespace(
    root=ROOT,
    raw=ROOT / "data" / "raw",
    reference=ROOT / "data" / "reference",
    processed=ROOT / "data" / "processed",
    modeling=ROOT / "data" / "modeling",
)

AIRPORT = CONFIG["airport"]
START_DATE = pd.Timestamp(CONFIG["start_date"])
END_DATE_EXCLUSIVE = pd.Timestamp(CONFIG["end_date_exclusive"])

TRACKED_AIRLINES = tuple(CONFIG["airlines"])
HORIZONS = tuple(CONFIG["horizons"])
FEATURES = CONFIG["features"]


In [2]:
BTS_ALIASES = {
    "dot_id_reporting_airline": "reporting_airline_id",
    "iata_code_reporting_airline": "iata_reporting_airline",
    "flight_number_reporting_airline": "flight_number",
    "dep_delay": "dep_delay_minutes",
    "dep_delay_minutes": "dep_delay_positive_minutes",
    "departure_delay_groups": "departure_delay_group",
    "taxi_out": "taxi_out_minutes",
    "taxi_in": "taxi_in_minutes",
    "crs_elapsed_time": "scheduled_elapsed_minutes",
    "actual_elapsed_time": "actual_elapsed_minutes",
    "air_time": "air_time_minutes",
    "distance": "distance_miles",
    "arr_delay": "arr_delay_minutes",
    "arr_delay_minutes": "arr_delay_positive_minutes",
    "arrival_delay_groups": "arrival_delay_group",
    "carrier_delay": "carrier_delay_minutes",
    "weather_delay": "weather_delay_minutes",
    "nas_delay": "nas_delay_minutes",
    "security_delay": "security_delay_minutes",
    "late_aircraft_delay": "late_aircraft_delay_minutes",
}

WEATHER_ALIASES = {
    "STATION": "noaa_station_id",
    "DATE": "weather_observed_utc",
    "temperature": "temperature_c",
    "dew_point_temperature": "dew_point_temperature_c",
    "relative_humidity": "relative_humidity_pct",
    "visibility": "visibility_km",
    "wind_speed": "wind_speed_mps",
    "wind_gust": "wind_gust_mps",
    "wind_direction": "wind_direction_degrees",
    "sea_level_pressure": "sea_level_pressure_hpa",
    "station_level_pressure": "station_level_pressure_hpa",
    "ceiling_height": "ceiling_height_m",
    "altimeter": "altimeter_hpa",
    "precipitation": "precipitation_mm",
}

WEATHER_BOUNDS = {
    "temperature_c": (-90.0, 60.0),
    "dew_point_temperature_c": (-100.0, 50.0),
    "relative_humidity_pct": (0.0, 100.0),
    "visibility_km": (0.0, 200.0),
    "wind_speed_mps": (0.0, 100.0),
    "wind_gust_mps": (0.0, 120.0),
    "wind_direction_degrees": (0.0, 360.0),
    "sea_level_pressure_hpa": (850.0, 1100.0),
    "station_level_pressure_hpa": (450.0, 1100.0),
    "ceiling_height_m": (0.0, 30000.0),
    "altimeter_hpa": (850.0, 1100.0),
    "precipitation_mm": (0.0, 500.0),
}

FAA_ALIASES = {
    "tail_number_clean": "tail_number_key",
    "year_mfr": "aircraft_year_manufactured",
    "type_aircraft": "aircraft_type_code",
    "type_engine": "engine_type_code",
    "mfr": "aircraft_manufacturer",
    "model": "aircraft_model",
    "no_eng": "number_of_engines",
    "no_seats": "number_of_seats",
    "ac_weight": "aircraft_weight_class",
    "speed": "aircraft_cruise_speed",
    "mfr_eng": "engine_manufacturer",
    "model_eng": "engine_model",
    "thrust": "engine_thrust",
}

### Raw source checks

In [3]:
RAW_BTS_PATH = PATHS.raw / "bts" / "bts_reporting_carrier_2022-01_to_2026-05.parquet"
RAW_FAA_PATH = PATHS.raw / "faa" / "faa_registry_with_deregistered.parquet"
RAW_NOAA_FILES = sorted((PATHS.raw / "noaa").glob("*/*.parquet"))

raw_file_summary = pd.DataFrame([
    {
        "source": "BTS flights",
        "files": 1,
        "rows_from_metadata": pq.ParquetFile(RAW_BTS_PATH).metadata.num_rows,
        "path_sample": RAW_BTS_PATH.relative_to(PATHS.root).as_posix(),
    },
    {
        "source": "NOAA hourly weather",
        "files": len(RAW_NOAA_FILES),
        "rows_from_metadata": sum(pq.ParquetFile(path).metadata.num_rows for path in RAW_NOAA_FILES),
        "path_sample": RAW_NOAA_FILES[0].relative_to(PATHS.root).as_posix(),
    },
    {
        "source": "FAA registry",
        "files": 1,
        "rows_from_metadata": pq.ParquetFile(RAW_FAA_PATH).metadata.num_rows,
        "path_sample": RAW_FAA_PATH.relative_to(PATHS.root).as_posix(),
    },
])
raw_file_summary


,source,files,rows_from_metadata,path_sample
0,BTS flights,1,1208677,data/raw/bts/bts_reporting_carrier_2022-01_to_...
1,NOAA hourly weather,5,87861,data/raw/noaa/2022/GHCNh_USW00023234_2022.parquet
2,FAA registry,1,533606,data/raw/faa/faa_registry_with_deregistered.pa...


### BTS Preprocessing

In [4]:
raw_bts = pd.read_parquet(RAW_BTS_PATH)
display(raw_bts)


,year,month,flight_date,reporting_airline,dot_id_reporting_airline,iata_code_reporting_airline,tail_number,flight_number_reporting_airline,origin_airport_id,origin,...,actual_elapsed_time,air_time,distance,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay,source_month,tail_number_clean
0,2022,1,2022-01-01,OO,20304,OO,NaN,4675.0,10140,ABQ,...,NaN,NaN,896.0,NaN,NaN,NaN,NaN,NaN,2022-01,<NA>
1,2022,1,2022-01-01,OO,20304,OO,N454SW,5949.0,10140,ABQ,...,145.0,130.0,896.0,NaN,NaN,NaN,NaN,NaN,2022-01,N454SW
2,2022,1,2022-01-01,OO,20304,OO,N203SY,4655.0,10157,ACV,...,65.0,47.0,250.0,0.0,0.0,0.0,0.0,171.0,2022-01,N203SY
3,2022,1,2022-01-01,OO,20304,OO,N146SY,5313.0,10157,ACV,...,85.0,47.0,250.0,NaN,NaN,NaN,NaN,NaN,2022-01,N146SY
4,2022,1,2022-01-01,OO,20304,OO,N788SK,4630.0,10372,ASE,...,126.0,112.0,848.0,NaN,NaN,NaN,NaN,NaN,2022-01,N788SK
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1208672,2026,5,2026-05-31,UA,19977,UA,N77584,2051.0,15304,TPA,...,334.0,314.0,2393.0,NaN,NaN,NaN,NaN,NaN,2026-05,N77584
1208673,2026,5,2026-05-31,UA,19977,UA,N37542,2839.0,15304,TPA,...,332.0,305.0,2393.0,0.0,0.0,46.0,0.0,0.0,2026-05,N37542
1208674,2026,5,2026-05-31,OO,20304,OO,N782SK,4683.0,15376,TUS,...,134.0,114.0,751.0,0.0,0.0,22.0,0.0,0.0,2026-05,N782SK
1208675,2026,5,2026-05-31,OO,20304,OO,N89362,5583.0,15376,TUS,...,162.0,118.0,751.0,15.0,0.0,0.0,0.0,0.0,2026-05,N89362


In [5]:
bts_key_columns = [
    "flight_date", "iata_code_reporting_airline", "flight_number_reporting_airline",
    "origin", "dest", "crs_dep_time", "dep_time", "crs_arr_time", "arr_time",
    "dep_delay", "arr_delay", "cancelled", "diverted", "tail_number",
    "carrier_delay", "weather_delay", "nas_delay", "security_delay", "late_aircraft_delay",
]
bts_key_columns = [column for column in bts_key_columns if column in raw_bts]

bts_missingness = pd.DataFrame(
    (100 * raw_bts[bts_key_columns].isna().mean())
    .round(2)
    .rename("missing_percent")
)
display(bts_missingness)

,missing_percent
flight_date,0.00
iata_code_reporting_airline,0.00
flight_number_reporting_airline,0.00
origin,0.00
dest,0.00
crs_dep_time,0.00
dep_time,1.15
crs_arr_time,0.00
arr_time,1.24
dep_delay,1.15


### NOAA Preprocessing

In [6]:
raw_noaa_weather = pd.concat(
    [pd.read_parquet(path, columns=[column for column in WEATHER_ALIASES if column in pq.ParquetFile(path).schema.names]) for path in RAW_NOAA_FILES],
    ignore_index=True,
)

display(raw_noaa_weather)


,STATION,DATE,temperature,dew_point_temperature,relative_humidity,visibility,wind_speed,wind_gust,wind_direction,sea_level_pressure,station_level_pressure,ceiling_height,altimeter,precipitation
0,USW00023234,2022-01-01T00:00:00,11.7,4.4,61,16.0,7.7,NaN,290,1012.4,1009.4,22000,NaN,NaN
1,USW00023234,2022-01-01T00:56:00,10.6,5.0,69,16.093,7.2,NaN,300,1012.8,1012.3,22000,1012.9,0.0
2,USW00023234,2022-01-01T01:56:00,10.0,4.4,68,16.093,5.1,NaN,300,1013.3,1012.6,22000,1013.2,0.0
3,USW00023234,2022-01-01T02:56:00,9.4,4.4,71,16.093,5.1,NaN,290,1014.0,1013.6,22000,1014.2,0.0
4,USW00023234,2022-01-01T03:56:00,9.4,5.0,74,16.093,5.1,NaN,310,1014.6,1014.0,22000,1014.6,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87856,USW00023234,2026-05-31T23:50:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
87857,USW00023234,2026-05-31T23:53:00,17.8,10.0,60,16.0,9.8,NaN,300,1014.2,1011.1,NaN,NaN,NaN
87858,USW00023234,2026-05-31T23:54:00,17.8,10.0,60,16.0,9.8,NaN,300,1014.2,1011.1,NaN,NaN,NaN
87859,USW00023234,2026-05-31T23:55:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
noaa_key_columns = [column for column in WEATHER_ALIASES if column in raw_noaa_weather]
noaa_value_columns = [column for column in list(WEATHER_ALIASES.keys())[2:] if column in raw_noaa_weather]

noaa_missingness = pd.DataFrame(
    (100 * raw_noaa_weather[noaa_value_columns].isna().mean())
    .sort_values(ascending=False)
    .round(2)
    .rename("missing_percent")
)

display(noaa_missingness)

,missing_percent
wind_gust,93.07
precipitation,62.30
altimeter,49.84
ceiling_height,48.32
station_level_pressure,46.89
sea_level_pressure,44.11
wind_direction,38.23
wind_speed,38.21
visibility,38.20
dew_point_temperature,37.79


### FAA Preprocessing


In [8]:
raw_faa = pd.read_parquet(RAW_FAA_PATH)

display(raw_faa)

,n_number,year_mfr,type_aircraft,type_aircraft_ref,type_engine,type_engine_ref,mfr_mdl_code,eng_mfr_mdl,mfr,model,...,air_worth_date,cancel_date,last_action_date,cert_issue_date,expiration_date,mode_s_code,unique_id,faa_record_type,tail_number_clean,faa_match
0,1,,4,4,1,1,0191006,,AERONCA,0-58B,...,,19480512,,19450601,<NA>,<NA>,<NA>,DEREG,N1,True
1,10,,4,4,1,1,1150538,,BEECH,D17S,...,,19470630,,19460812,<NA>,<NA>,<NA>,DEREG,N10,True
2,100,1940,4,4,1,1,7100510,17003,PIPER,J3C-65,...,19540430,NaN,20230122,20050506,20270430,50002263,00600060,MASTER,N100,True
3,1000,,4,4,1,1,05635B0,,DOUGLAS,M-3,...,,19340723,,,<NA>,<NA>,<NA>,DEREG,N1000,True
4,10000,,4,4,1,1,2130004,,CIRRUS DESIGN CORP,SR22T,...,,NaN,20240823,20240823,20310831,50003445,01443200,MASTER,N10000,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
533601,9ZU,1959,4,4,1,1,7101828,41508,PIPER,PA-18-150,...,20011010,NaN,20230401,20171114,20271130,53066314,00264074,MASTER,N9ZU,True
533602,9ZV,,6,6,3,3,1181511,03020,BELL,206B,...,20010710,NaN,20250726,20250726,20320731,53066315,00281264,MASTER,N9ZV,True
533603,9ZW,,4,4,1,1,05608IF,,CARPENTER SCOTT DAVID,S-9,...,,20110511,20000106,,<NA>,<NA>,<NA>,DEREG,N9ZW,True
533604,9ZX,1986,4,4,1,1,05655US,09050,POBEREZNY PAUL H,HIPERLIGHT SNS-8,...,19860731,NaN,20230310,20230310,20300331,53066317,00272135,MASTER,N9ZX,True


In [9]:
faa_key_columns = [
    "tail_number_clean", "faa_record_type", "year_mfr", "type_aircraft",
    "type_engine", "mfr", "model", "no_eng", "no_seats", "ac_weight", "speed", "thrust",
]
faa_key_columns = [column for column in faa_key_columns if column in raw_faa]

faa_missingness = pd.DataFrame(
    (100 * raw_faa[faa_key_columns].isna().mean())
    .round(2)
    .rename("missing_percent")
)

display(faa_missingness)

,missing_percent
tail_number_clean,0.00
faa_record_type,0.00
year_mfr,0.00
type_aircraft,0.00
type_engine,0.00
mfr,0.00
model,0.00
no_eng,0.00
no_seats,0.00
ac_weight,0.00


### Helper functions

In [10]:
def _hhmm_to_minutes(values: pd.Series) -> pd.Series:
    """Converts HHMM (1230) into minutes since midnight"""
    numeric = pd.to_numeric(values, errors="coerce")
    hours = np.floor(numeric / 100)
    minutes = numeric % 100
    valid = numeric.notna() & minutes.lt(60) & hours.le(24)
    return (hours * 60 + minutes).where(valid)

def _local_to_utc(dates: pd.Series, times: pd.Series, airports: pd.Series, timezone_lookup: dict[str, str]) -> pd.Series:
    """Convert airport local date and times into UTC timestamps"""
    naive = pd.to_datetime(dates, errors="coerce") + pd.to_timedelta(_hhmm_to_minutes(times), unit="m")
    output = pd.Series(pd.NaT, index=dates.index, dtype="datetime64[ns, UTC]")
    for airport, index in airports.groupby(airports, observed=True).groups.items():
        timezone_name = timezone_lookup.get(str(airport))
        if not timezone_name:
            continue
        localized = naive.loc[index].dt.tz_localize(
            timezone_name, ambiguous="NaT", nonexistent="shift_forward"
        )
        output.loc[index] = localized.dt.tz_convert("UTC")
    return output

### Normalize each source

In [11]:
def normalize_bts(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize SFO flight data from the BTS pull."""
    output = df.rename(columns=BTS_ALIASES).copy()
    output.columns = [str(column).strip() for column in output.columns]
    output["flight_date"] = pd.to_datetime(output["flight_date"], errors="coerce")

    output = output.loc[output["flight_date"].ge(START_DATE) & output["flight_date"].lt(END_DATE_EXCLUSIVE)].copy()
    
    for column in ("origin", "dest", "reporting_airline", "tail_number"):
        output[column] = output[column].astype("string").str.strip().str.upper()
    
    airport_touch = output["origin"].eq(AIRPORT) | output["dest"].eq(AIRPORT)
    output = output.loc[airport_touch].copy()
    output["day_of_week"] = output["flight_date"].dt.dayofweek.add(1).astype("int8")
    output["day_of_month"] = output["flight_date"].dt.day.astype("int8")

    numeric_columns = [
        "crs_dep_time", "dep_time", "crs_arr_time", "arr_time", "dep_delay_minutes",
        "dep_delay_positive_minutes", "arr_delay_minutes", "arr_delay_positive_minutes",
        "cancelled", "diverted", "scheduled_elapsed_minutes", "actual_elapsed_minutes",
        "air_time_minutes", "distance_miles", "carrier_delay_minutes",
        "weather_delay_minutes", "nas_delay_minutes", "security_delay_minutes",
        "late_aircraft_delay_minutes", "taxi_out_minutes", "taxi_in_minutes",
    ]
    for column in numeric_columns:
        if column in output:
            output[column] = pd.to_numeric(output[column], errors="coerce")

    output["cancelled"] = output["cancelled"].fillna(0).astype("int8")
    output["diverted"] = output["diverted"].fillna(0).astype("int8")

    timezone_lookup = {code: record["tz"] for code, record in airportsdata.load("IATA").items()}

    output["scheduled_dep_utc"] = _local_to_utc(output["flight_date"], output["crs_dep_time"], output["origin"], timezone_lookup)
    
    scheduled_arr_clock = _local_to_utc(output["flight_date"], output["crs_arr_time"], output["dest"], timezone_lookup)
    
    fallback_roll = scheduled_arr_clock.copy()
    rolls = fallback_roll.lt(output["scheduled_dep_utc"] - pd.Timedelta(hours=1))
    fallback_roll.loc[rolls] = fallback_roll.loc[rolls] + pd.Timedelta(days=1)
    
    elapsed = pd.to_numeric(output["scheduled_elapsed_minutes"], errors="coerce")
    derived_arrival = output["scheduled_dep_utc"] + pd.to_timedelta(elapsed, unit="m")
    
    output["scheduled_arr_utc"] = derived_arrival.where(elapsed.notna(), fallback_roll)
    
    output["actual_dep_utc"] = output["scheduled_dep_utc"] + pd.to_timedelta(output["dep_delay_minutes"], unit="m")
    output["actual_arr_utc"] = output["scheduled_arr_utc"] + pd.to_timedelta(output["arr_delay_minutes"], unit="m")

    output.loc[output["cancelled"].eq(1), ["actual_dep_utc", "actual_arr_utc"]] = pd.NaT
    
    output["tail_number_key"] = (output["tail_number"].astype("string").str.removeprefix("N"))

    # Creating our own flight ID
    output["flight_id"] = (
        output["flight_date"].dt.strftime("%Y%m%d")
        + "_" + output["reporting_airline"].fillna("UNK")
        + "_" + output["flight_number"].astype("string").fillna("UNK")
        + "_" + output["origin"].fillna("UNK")
        + "_" + output["dest"].fillna("UNK")
        + "_" + output["crs_dep_time"].astype("Int64").astype("string").fillna("UNK")
    )
    if output["flight_id"].duplicated().any():
        duplicates = int(output["flight_id"].duplicated().sum())
        raise ValueError(f"Custom BTS flight_id is not unique ({duplicates} duplicates)")
    
    return output.sort_values(["scheduled_dep_utc", "flight_id"]).reset_index(drop=True)

def normalize_weather_file(path: Path) -> pd.DataFrame:
    """Keep valid SFO weather values within the modeling period."""
    raw_names = list(WEATHER_ALIASES)
    available = set(pd.read_parquet(path, columns=[]).columns)
    columns = [column for column in raw_names if column in available]

    # Handles pyarrow parquet column reading issues
    if not columns:
        available = set(pq.ParquetFile(path).schema.names)
        columns = [column for column in raw_names if column in available]

    df = pd.read_parquet(path, columns=columns)
    output = pd.DataFrame(index=df.index)
    output["noaa_station_id"] = df["STATION"].astype("string").str.strip()
    output["weather_observed_utc"] = pd.to_datetime(df["DATE"], errors="coerce", utc=True)
    for raw_name, canonical in list(WEATHER_ALIASES.items())[2:]:
        if raw_name not in df:
            output[canonical] = np.nan
            continue
        numeric = pd.to_numeric(df[raw_name], errors="coerce")
        lower, upper = WEATHER_BOUNDS[canonical]
        output[canonical] = numeric.where(numeric.between(lower, upper))

    cutoff = END_DATE_EXCLUSIVE.tz_localize("UTC")
    output = output.loc[output["weather_observed_utc"].ge(START_DATE.tz_localize("UTC")) & output["weather_observed_utc"].lt(cutoff)].copy()
    
    weather_columns = list(WEATHER_BOUNDS)
    output = output.loc[~output[weather_columns].isna().all(axis=1)]

    return output.dropna(subset=["weather_observed_utc"]).drop_duplicates(["noaa_station_id", "weather_observed_utc"], keep="last")

def normalize_faa(df: pd.DataFrame) -> pd.DataFrame:
    """Standardize FAA types and retain one record per aircraft."""
    output = df.rename(columns=FAA_ALIASES).copy()
    numeric_columns = [
        "aircraft_year_manufactured", "number_of_engines",
        "number_of_seats", "aircraft_cruise_speed", "engine_thrust",
    ]
    for column in numeric_columns:
        if column in output:
            output[column] = pd.to_numeric(output[column], errors="coerce")
    output["tail_number_key"] = (output["tail_number_key"].astype("string").str.strip().str.upper().str.removeprefix("N"))
    output = output.dropna(subset=["tail_number_key"])
    output = output.loc[output["tail_number_key"].ne("")].copy()
    if "faa_record_type" in output:
        priority = output["faa_record_type"].map({"MASTER": 0, "DEREG": 1}).fillna(2)
        output = output.assign(_record_priority=priority).sort_values(
            ["tail_number_key", "_record_priority"], kind="stable"
        ).drop(columns="_record_priority")
    return output.drop_duplicates("tail_number_key", keep="first").reset_index(drop=True)

### Process Datasets

The three normalized tables are saved once for the merging and feature engineering notebook.

In [12]:

def process_all_sources():
    """Normalize and write the three processed source tables."""
    PATHS.processed.mkdir(parents=True, exist_ok=True)

    flights = normalize_bts(pd.read_parquet(RAW_BTS_PATH))
    flights.to_parquet(PATHS.processed / "flights_sfo_touch.parquet", index=False)

    weather_files = [normalize_weather_file(path) for path in RAW_NOAA_FILES]
    weather = pd.concat(weather_files, ignore_index=True)
    weather = weather.sort_values(["noaa_station_id", "weather_observed_utc"])
    weather = weather.drop_duplicates(["noaa_station_id", "weather_observed_utc"], keep="last")
    weather.to_parquet(PATHS.processed / "weather_hourly.parquet", index=False)

    aircraft = normalize_faa(pd.read_parquet(RAW_FAA_PATH))
    aircraft.to_parquet(PATHS.processed / "faa_aircraft.parquet", index=False)

    coverage = f"{START_DATE.date()} to {(END_DATE_EXCLUSIVE - pd.Timedelta(days=1)).date()}"
    processed_summary = [
        {"table": "flights", "rows": len(flights), "coverage": coverage},
        {"table": "weather", "rows": len(weather), "coverage": coverage},
        {"table": "aircraft", "rows": len(aircraft), "coverage": f"{aircraft.tail_number_key.nunique():,} unique tails"},
    ]
    return pd.DataFrame(processed_summary)


### Preprocessing Result

In [13]:
processed_summary = process_all_sources()
display(processed_summary)


,table,rows,coverage
0,flights,1208677,2022-01-01 to 2026-05-31
1,weather,54660,2022-01-01 to 2026-05-31
2,aircraft,533606,"533,606 unique tails"


In [14]:
processed_flights = pd.read_parquet(PATHS.processed / "flights_sfo_touch.parquet")
processed_weather = pd.read_parquet(PATHS.processed / "weather_hourly.parquet")
processed_aircraft = pd.read_parquet(PATHS.processed / "faa_aircraft.parquet")

preprocessing_result = pd.DataFrame([
    {
        "source": "BTS flights",
        "raw_rows": len(raw_bts),
        "processed_rows": len(processed_flights),
    },
    {
        "source": "NOAA hourly weather",
        "raw_rows": raw_file_summary.loc[raw_file_summary["source"].eq("NOAA hourly weather"), "rows_from_metadata"].iat[0],
        "processed_rows": len(processed_weather),
    },
    {
        "source": "FAA registry",
        "raw_rows": len(raw_faa),
        "processed_rows": len(processed_aircraft),
    },
])
preprocessing_result["rows_removed"] = preprocessing_result["raw_rows"] - preprocessing_result["processed_rows"]
preprocessing_result["rows_kept_percent"] = (100 * preprocessing_result["processed_rows"] / preprocessing_result["raw_rows"]).round(2)
display(preprocessing_result)

,source,raw_rows,processed_rows,rows_removed,rows_kept_percent
0,BTS flights,1208677,1208677,0,100.00
1,NOAA hourly weather,87861,54660,33201,62.21
2,FAA registry,533606,533606,0,100.00


### BTS After Preprocessing

In [15]:
bts_processed_columns = processed_flights.columns

bts_missingness = pd.DataFrame(
    (100 * processed_flights[bts_processed_columns].isna().mean())
    .sort_values(ascending=False)
    .round(2)
    .rename("missing_percent")
)

display(bts_missingness)

,missing_percent
cancellation_code,98.80
carrier_delay_minutes,78.28
late_aircraft_delay_minutes,78.28
weather_delay_minutes,78.28
nas_delay_minutes,78.28
security_delay_minutes,78.28
arr_delay_minutes,1.41
arr_del15,1.41
arrival_delay_group,1.41
actual_elapsed_minutes,1.41


### NOAA After Preprocessing

In [16]:
noaa_processed_columns = processed_weather.columns

noaa_missingness = pd.DataFrame(
    (100 * processed_weather[noaa_processed_columns].isna().mean())
    .sort_values(ascending=False)
    .round(2)
    .rename("missing_percent")
)

display(noaa_missingness)

,missing_percent
wind_gust_mps,88.87
precipitation_mm,39.40
altimeter_hpa,19.38
ceiling_height_m,16.93
station_level_pressure_hpa,14.63
wind_direction_degrees,10.53
sea_level_pressure_hpa,10.16
wind_speed_mps,0.68
visibility_km,0.67
dew_point_temperature_c,0.01


### FAA After Preprocessing

In [17]:
faa_processed_columns = processed_aircraft.columns

faa_missingness = pd.DataFrame(
    (100 * processed_aircraft[faa_processed_columns].isna().mean())
    .sort_values(ascending=False)
    .round(2)
    .rename("missing_percent")
)

display(faa_missingness)

,missing_percent
cancel_date,59.00
mode_s_code,41.00
expiration_date,41.00
unique_id,41.00
aircraft_year_manufactured,30.14
engine_model,24.63
horsepower,24.63
engine_manufacturer,24.63
engine_thrust,24.63
type_aircraft_ref,0.00
